# MTTV-FLP — Chat avec Qwen2.5-7B-Instruct (GPU T4)

**Modèle** : [`Qwen/Qwen2.5-7B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-7B-Instruct)

**Exécution** :
1. `Exécution` → `Modifier le type d'exécution` → `T4 GPU`
2. `Exécution` → `Tout exécuter`

**Contraintes** :
- Le modèle charge ~14 Go en float16 sur le T4 (16 Go VRAM)
- `device_map="cuda:0"` garantit que TOUT le modèle est sur GPU
- Si le GPU n'est pas activé, le notebook plante avec un message clair

---


In [1]:
"""
CELLULE 1 — Installation des dépendances
Exécutée en premier pour éviter les ModuleNotFoundError.
"""
import subprocess
import sys

print("=" * 60)
print("CELLULE 1/5 — Installation des dépendances")
print("=" * 60)

DEPS = [
    "transformers>=4.40.0",
    "accelerate>=0.28.0",
    "bitsandbytes>=0.43.0",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + DEPS)

# Vérification
import importlib.metadata
for dep in ["transformers", "accelerate", "bitsandbytes", "torch"]:
    try:
        v = importlib.metadata.version(dep)
        print(f"  [OK] {dep}=={v}")
    except importlib.metadata.PackageNotFoundError:
        print(f"  [FAIL] {dep} NON TROUVÉ")

print("[OK] Dépendances installées avec succès")


CELLULE 1/5 — Installation des dépendances
  [OK] transformers==5.12.1
  [OK] accelerate==1.14.0
  [OK] bitsandbytes==0.49.2
  [OK] torch==2.11.0+cu128
[OK] Dépendances installées avec succès


In [2]:
"""
CELLULE 2 — Vérification GPU obligatoire
Plante avec un message clair si CUDA n'est pas disponible.
"""
import torch

print("=" * 60)
print("CELLULE 2/5 — Vérification GPU")
print("=" * 60)

if not torch.cuda.is_available():
    raise RuntimeError(
        "\n" + "!" * 60 + "\n"
        "  ERREUR : CUDA n'est pas disponible.\n"
        "  Le modèle Qwen2.5-7B-Instruct nécessite un GPU.\n"
        "\n"
        "  Solution :\n"
        "  1. Menu → Exécution → Modifier le type d'exécution\n"
        "  2. Sélectionner T4 GPU\n"
        "  3. Exécuter à nouveau cette cellule\n"
        "\n"
        "  Vérification : Exécutez !nvidia-smi pour confirmer\n"
        "!" * 60
    )

# Infos GPU
gpu_name = torch.cuda.get_device_name(0)
vram_total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
vram_used = torch.cuda.memory_allocated(0) / (1024**3)

print(f"  GPU détecté     : {gpu_name}")
print(f"  VRAM totale     : {vram_total:.1f} Go")
print(f"  VRAM utilisée   : {vram_used:.2f} Go")
print(f"  CUDA version    : {torch.version.cuda}")
print(f"  PyTorch version : {torch.__version__}")
print("[OK] GPU prêt — chargement du modèle possible")


CELLULE 2/5 — Vérification GPU
  GPU détecté     : Tesla T4
  VRAM totale     : 14.6 Go
  VRAM utilisée   : 0.00 Go
  CUDA version    : 12.8
  PyTorch version : 2.11.0+cu128
[OK] GPU prêt — chargement du modèle possible


In [3]:
"""
CELLULE 3 — Chargement du modèle Qwen2.5-7B-Instruct sur GPU

Points critiques :
  - device_map="cuda:0"  → force TOUT le modèle sur GPU (≠ "auto")
  - torch_dtype=torch.float16 → ~14 Go, tient dans 16 Go du T4
  - low_cpu_mem_usage=True → évite de dupliquer en RAM avant GPU
  - trust_remote_code=True → nécessaire pour Qwen2.5
"""
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

print("=" * 60)
print("CELLULE 3/5 — Chargement du modèle")
print("=" * 60)

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# ─── Tokenizer ────────────────────────────────────────────────────────
print(f"[1/3] Chargement du tokenizer {MODEL_NAME}...")
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    padding_side="right",
)
# Certains tokenizers Qwen n'ont pas de pad_token ; on utilise eos_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"  Tokenizer chargé en {time.time() - t0:.1f}s")
print(f"  Vocabulaire : {len(tokenizer)} tokens")
print(f"  Pad token   : {tokenizer.pad_token}")
print(f"  EOS token   : {tokenizer.eos_token}")

# ─── Modèle ───────────────────────────────────────────────────────────
print(f"[2/3] Chargement du modèle {MODEL_NAME} en float16 sur cuda:0...")
print(f"  (Ce chargement prend ~2-3 minutes sur T4)")
t0 = time.time()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="cuda:0",           # ← CRITIQUE : force GPU
    torch_dtype=torch.float16,       # ← 14 Go au lieu de 28 Go
    trust_remote_code=True,          # ← nécessaire pour Qwen
    low_cpu_mem_usage=True,          # ← évite duplication CPU
)

load_time = time.time() - t0
print(f"[3/3] Vérification du chargement")

# --- Vérification du device ------------------------------------------
print(f"  Modèle chargé en {load_time:.1f}s")
print(f"  model.device        : {model.device}")

# Vérifier que tous les paramètres sont bien sur CUDA
n_on_cpu = sum(1 for p in model.parameters() if p.device.type == "cpu")
n_on_cuda = sum(1 for p in model.parameters() if p.device.type == "cuda")
total_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"  Paramètres totaux   : {total_params:.2f}B")
print(f"  Couches sur CPU     : {n_on_cpu}")
print(f"  Couches sur CUDA    : {n_on_cuda}")

if n_on_cpu > 0:
    print("  ⚠️  Attention : certaines couches sont restées sur CPU !")
else:
    print(f"  ✅ Modèle entièrement sur GPU : {model.device}")

# --- VRAM après chargement -------------------------------------------
vram_after = torch.cuda.memory_allocated(0) / (1024**3)
print(f"  VRAM utilisée après chargement : {vram_after:.2f} Go / 16 Go")

# --- nvidia-smi ------------------------------------------------------
import subprocess as sp
print("\n  ─── nvidia-smi (VRAM) ───")
result = sp.run(
    ["nvidia-smi", "--query-gpu=memory.total,memory.used,memory.free",
     "--format=csv,noheader,nounits"],
    capture_output=True, text=True
)
print(f"  {result.stdout.strip()}")

print("[OK] Modèle prêt pour l'inférence")


CELLULE 3/5 — Chargement du modèle
[1/3] Chargement du tokenizer Qwen/Qwen2.5-7B-Instruct...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[3/3] Vérification du chargement
  Modèle chargé en 410.8s
  model.device        : cuda:0
  Paramètres totaux   : 7.62B
  Couches sur CPU     : 0
  Couches sur CUDA    : 339
  ✅ Modèle entièrement sur GPU : cuda:0
  VRAM utilisée après chargement : 14.19 Go / 16 Go

  ─── nvidia-smi (VRAM) ───
  15360, 14649, 264
[OK] Modèle prêt pour l'inférence


In [5]:
# CELLULE 4/5 — Fonction d'inférence corrigée
import torch

def generate(messages, max_new_tokens=256, temperature=0.7, top_p=0.9):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(response, skip_special_tokens=True)

# Test
messages = [
    {"role": "system", "content": "Tu es un assistant utile qui explique simplement."},
    {"role": "user", "content": "Explique la photosynthèse à un enfant de 10 ans, en 3 phrases."}
]

print("Exemple d'inférence...\n")
print("Prompt :", messages[1]["content"], "\n")
print(generate(messages))

Exemple d'inférence...

Prompt : Explique la photosynthèse à un enfant de 10 ans, en 3 phrases. 

La photosynthèse est comme une recette magique que les plantes utilisent pour se nourrir. Elles prennent le soleil, l'eau et l'air, et en font de la nourriture qui les fait grandir et de l'oxygène que nous pouvons respirer. En d'autres termes, c'est comment les plantes fabriquent leur propre pain avec du soleil !


In [ ]:
"""
CELLULE 5 — Boucle de chat interactive
"""

print("=" * 60)
print("CELLULE 5/5 — Chat interactif")
print("=" * 60)
print("  Tapez 'quit' / 'exit' / 'stop' pour quitter")
print("  Tapez 'clear' pour effacer l'écran")
print("=" * 60)
print()

history = []

while True:
    try:
        user_input = input("\n🧑 Vous > ")
    except EOFError:
        print("\nAu revoir !")
        break

    if user_input.lower() in ("quit", "exit", "stop"):
        print("\nAu revoir !")
        break

    if user_input.lower() == "clear":
        import os as _os
        _os.system("cls" if _os.name == "nt" else "clear")
        continue

    if not user_input.strip():
        continue

    # Génération
    print("\n🤖 Assistant > ", end="", flush=True)
    t0 = time.time()
    try:
        messages = [
            {"role": "system", "content": "Tu réponds toujours en français."},
            {"role": "user", "content": user_input}
        ]
        response = generate(messages, max_new_tokens=512, temperature=0.7)
        elapsed = time.time() - t0
        print(response)
        print(f"\n   ✨ {elapsed:.1f}s — {len(response.split())} mots")
        history.append((user_input, response))
    except Exception as e:
        print(f"\n  ❌ Erreur : {e}")

CELLULE 5/5 — Chat interactif
  Tapez 'quit' / 'exit' / 'stop' pour quitter
  Tapez 'clear' pour effacer l'écran


🧑 Vous > bonjour

🤖 Assistant > Bonjour ! Comment puis-je vous aider aujourd'hui ?

   ✨ 0.9s — 8 mots

🧑 Vous > 2+2=?

🤖 Assistant > 2+2 fait 4.

   ✨ 0.6s — 3 mots

🧑 Vous > quel est le sens de la vie ?

🤖 Assistant > La question du sens de la vie est extrêmement subjective et peut varier considérablement selon les individus, les cultures et les philosophies. Pour certains, le sens de la vie réside dans la quête de bonheur et d'accomplissement personnel. D'autres peuvent trouver leur sens dans les relations avec les autres, l'aide aux autres, ou dans des buts plus éthiques ou spirituels. Certains pensent que le sens de la vie n'est pas un "sens" unique et universel, mais plutôt une collection de significations individuelles que chaque personne crée pour elle-même. Il n'y a donc pas de réponse définitive à cette question, mais plutôt une exploration personnelle qui varie 